# EuroSAT on Colab GPU
Upload this notebook to https://colab.research.google.com/ . Select **Runtime > Change runtime type > GPU** before running cells in order.

Upload `outputs/colab-setup/landcover-colab.zip` when requested. The bundle contains current code and fixed splits; it contains no imagery or checkpoints. The imagery ZIP is cached in Drive after the first download, then copied to local Colab storage for extraction and training.

This starts a **new 63-run CUDA study** with the same 3,000-step settings. Your Mac study stays separate. Allow **18–23 GB of free Google Drive space** for results and the dataset ZIP cache. Checkpoints and graphs save directly to Drive; images are read from local Colab storage. Drive checkpoint writes can add overhead.

Run cells individually. GPU availability and runtime limits vary; this is not a promise that all runs fit in one session. Reopen this same notebook and rerun setup after a disconnect. Completed jobs are verified and skipped; an interrupted job restarts from scratch in a fresh attempt. Do not change the study name or upload a different bundle midway through a study.

Sources: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [official PyTorch 2.6 CUDA installation](https://pytorch.org/get-started/previous-versions/#v260).


In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive, files

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/CS4343")
DRIVE.mkdir(exist_ok=True)
BUNDLE = DRIVE / "landcover-colab.zip"
if not BUNDLE.exists():
    uploaded = files.upload()
    if "landcover-colab.zip" not in uploaded:
        raise ValueError("Upload landcover-colab.zip from outputs/colab-setup.")
    BUNDLE.write_bytes(uploaded["landcover-colab.zip"])
ROOT = Path("/content/cs4343-landcover")
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for name in archive.namelist():
        if not (ROOT / name).resolve().is_relative_to(ROOT.resolve()):
            raise ValueError("Unsafe bundle path")
    archive.extractall(ROOT)
metadata = json.loads((ROOT / "bundle-provenance.json").read_text())
for name, expected in metadata["files"].items():
    if hashlib.sha256((ROOT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Bundle hash mismatch: {name}")
os.chdir(ROOT)
os.environ.update(
    PYTHONPATH=str(ROOT / "src"),
    PYTHONUNBUFFERED="1",
    TORCH_HOME=str(ROOT / ".cache/torch"),
    MPLCONFIGDIR=str(ROOT / ".cache/matplotlib"),
    MPLBACKEND="Agg",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
)
STUDY = DRIVE / "outputs/cuda-study"
print("Code:", ROOT, "\nPersistent results:", STUDY)

## Install the isolated CUDA environment
Do not use `uv sync` here: the project's Linux lock uses CPU-only PyTorch. This cell preserves `uv.lock`, exports its other pinned dependencies, and uses official CUDA 12.4 builds of the same PyTorch/torchvision versions. Environment details are saved with the study.


In [ ]:
def command(*args):
    args = [str(arg) for arg in args]
    with subprocess.Popen(
        args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            returncode = process.wait()
        except BaseException:
            process.kill()
            process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, args)


command(sys.executable, "-m", "pip", "install", "-q", "uv")
command("uv", "python", "install", "3.12")
PYTHON = ROOT / ".venv-colab/bin/python"
if not PYTHON.exists():
    command("uv", "venv", "--python", "3.12", ROOT / ".venv-colab")
command("uv", "pip", "install", "--python", PYTHON, "-r", "configs/colab-requirements.txt")
command(
    "uv",
    "pip",
    "install",
    "--python",
    PYTHON,
    "torch==2.6.0+cu124",
    "torchvision==0.21.0+cu124",
    "--constraint",
    "configs/colab-requirements.txt",
    "--index-url",
    "https://download.pytorch.org/whl/cu124",
)
command(
    PYTHON,
    "-c",
    "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'; "
    "print(torch.__version__, torch.cuda.get_device_name(0)); "
    "x=torch.ones(32,32,device='cuda'); print('CUDA check:', (x@x).mean().item())",
)

## Download and verify EuroSAT
The first session downloads about 2.1 GB from the publisher and saves a verified ZIP to `MyDrive/CS4343/datasets/EuroSAT_MS.zip`. Later runtimes copy that ZIP from Drive to local disk, then extract locally; copying still takes time, but avoids downloading again from the publisher. Rerunning within the same runtime reuses the local ZIP and completed extraction. An existing local download is cached without downloading it again. Both cache copies and the downloader verify the publisher's MD5. Study preparation checks image hashes; the shared splits never change. Interrupted copies keep a `.partial` file and retry on the next run.


In [ ]:
command(
    PYTHON,
    "-c",
    """
import hashlib
import shutil
import sys
from pathlib import Path
from landcover.download import ARCHIVES, download
cache = Path(sys.argv[1]) / 'datasets' / 'EuroSAT_MS.zip'
local = Path('data/raw/EuroSAT_MS.zip')
local.parent.mkdir(parents=True, exist_ok=True)
cache.parent.mkdir(parents=True, exist_ok=True)

def verified_copy(source, destination):
    partial = destination.with_suffix('.partial')
    shutil.copyfile(source, partial)
    with partial.open('rb') as stream:
        if hashlib.file_digest(stream, 'md5').hexdigest() != ARCHIVES['ms'][1]:
            raise ValueError(f'Archive cache checksum mismatch: {partial}')
    partial.replace(destination)

if not local.exists() and cache.exists():
    print('Restoring EuroSAT ZIP from Drive to local disk...', flush=True)
    verified_copy(cache, local)
elif local.exists():
    print('Reusing local EuroSAT ZIP.', flush=True)
else:
    print('Downloading EuroSAT from publisher once...', flush=True)
root = download('ms')
if not cache.exists():
    print('Saving verified EuroSAT ZIP to Drive for future sessions...', flush=True)
    verified_copy(local, cache)
print('Persistent archive:', cache, flush=True)
candidates = [p.parent for p in root.rglob('AnnualCrop') if p.is_dir()]
if len(candidates) != 1:
    raise ValueError(f'Expected one EuroSAT class root, found {candidates}')
target = Path('EuroSAT_MS')
if not target.exists():
    target.symlink_to(candidates[0].resolve(), target_is_directory=True)
elif target.resolve() != candidates[0].resolve():
    raise ValueError('Existing EuroSAT_MS points to different data')
print('Local imagery:', target.resolve())
""",
    DRIVE,
)

## Freeze or verify the CUDA study
This audits data and records the environment; it does not train yet. Same splits, seeds, steps, batches and validation schedule as the Mac study. Device changes from MPS to CUDA; the completed Mac runs are not merged into this study.


In [ ]:
command(
    PYTHON,
    "-c",
    """
import json
import subprocess
import sys
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
import torch
from landcover.config import Config
from landcover.study import prepare_study, load_study
root = Path(sys.argv[1])
base = replace(Config.load('configs/final-study.yaml'), device='cuda', output=str(root))
if not root.exists():
    prepare_study(root, base)
load_study(root)
record = json.loads(Path('bundle-provenance.json').read_text())
record.update(device='cuda', gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda,
              time=datetime.now(timezone.utc).isoformat(),
              packages=subprocess.check_output(
                  ['uv', 'pip', 'freeze', '--python', sys.executable], text=True))
sessions = root / 'sessions'
sessions.mkdir(exist_ok=True)
name = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
(sessions / (name + '.json')).write_text(json.dumps(record, indent=2))
print('Ready: 63 runs; completed runs will be skipped.')
""",
    STUDY,
)

## Train and generate validation results
This is the long-running cell. The first full-data run is a useful timing reference, but smaller fractions have less evaluation work. No reliable total runtime is known until this GPU is measured. Keep this notebook connected; do not run the study from two sessions at once.

The checkpoint files and completed runs already live in Drive. On interruption, rerun setup and this cell; `--retry-incomplete` retains failed attempts and starts the unfinished job again. It does not resume the last optimizer step.


In [ ]:
import time

started = time.monotonic()
command(PYTHON, "-m", "landcover.cli", "study-run", "--study", STUDY, "--retry-incomplete")
print(
    f"Training/validation finished in {(time.monotonic() - started) / 3600:.2f} hours this session."
)

## View validation graphs and tables
The full comparison requires all 63 jobs. Individual `learning_curves.png` files appear in each run folder when that run finishes. All artifacts are in Google Drive under `CS4343/outputs/cuda-study`.


In [ ]:
import csv

from IPython.display import Image, display

report = STUDY / "report-val"
if not report.exists():
    print("Full report is not ready yet. Completed run graphs:")
    for path in sorted(STUDY.glob("runs/*/attempt-*/learning_curves.png")):
        print(path)
else:
    for path in sorted(report.glob("*.png")):
        print(path.name)
        display(Image(filename=str(path)))
    with (report / "summary.csv").open() as stream:
        for row in csv.DictReader(stream):
            print(row)

## Final test phase — run only after all training is finished
This uses held-out test data. Freeze the comparison protocol first and do not change settings based on these scores. Change `RUN_FINAL_TEST` to `True` when ready. Test evaluation and its graphs save to `report-test`.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    command(PYTHON, "-m", "landcover.cli", "study-test", "--study", STUDY, "--allow-test")
    for path in sorted((STUDY / "report-test").glob("*.png")):
        display(Image(filename=str(path)))
else:
    print("Test data left untouched. Finish all validation runs first.")